In [1]:
import pandas as pd
import numpy as np

In [2]:
products = pd.read_csv('/content/combined_products_2025_v2-2.csv')

In [75]:
suppliers = pd.read_csv('/content/synthetic_suppliers.csv')

In [24]:
supplier_product = pd.read_csv('/content/suppliers_products.csv')

In [4]:
products.head()

,date,ppi_value,country_code,ppi_source,real_price,product,ppi_region,year,month
0,1976-01-01,100.0,ARE,FRED,0.26,integrated_circuit_components,OAC,1976,1
1,1976-02-01,100.0,ARE,FRED,0.26,integrated_circuit_components,OAC,1976,2
2,1976-03-01,100.0,ARE,FRED,0.26,integrated_circuit_components,OAC,1976,3
3,1976-04-01,100.0,ARE,FRED,0.26,integrated_circuit_components,OAC,1976,4
4,1976-05-01,100.0,ARE,FRED,0.26,integrated_circuit_components,OAC,1976,5


In [6]:
products['product'].unique()

array(['integrated_circuit_components', 'microprocessors',
       'power_devices', 'transistors'], dtype=object)

# Defining a base annual demand per product

In [7]:
base_annual_demand = {
    'microprocessors': 180000,
    'power_devices': 300000,
    'transistors': 2400000,
    'integrated_circuit_components': 4800000
}

# Generating Monthly Demand with Seasonality and Noise

In [8]:
months = pd.date_range('2024-01-01', '2025-12-01', freq='MS')
records = []

for product, annual in base_annual_demand.items():
  monthly_mean = annual / 12
  for date in months:
    season_factor = 1 + 0.15 * np.sin(2 * np.pi * (date.month / 12))
    demand = np.random.normal(monthly_mean * season_factor, monthly_mean * 0.1)
    demand = max(int(demand), 0)
    records.append({
        'date': date,
        'product': product,
        'monthly_demand_units': demand
    })
demand_df = pd.DataFrame(records)

In [14]:
products.groupby('country_code')['product'].value_counts().tail()

country_code  product                      
THA           microprocessors                  535
              power_devices                    167
USA           integrated_circuit_components    600
              transistors                      588
              microprocessors                  535
Name: count, dtype: int64

# Unit Values

In [15]:
products.head()

,date,ppi_value,country_code,ppi_source,real_price,product,ppi_region,year,month
0,1976-01-01,100.0,ARE,FRED,0.26,integrated_circuit_components,OAC,1976,1
1,1976-02-01,100.0,ARE,FRED,0.26,integrated_circuit_components,OAC,1976,2
2,1976-03-01,100.0,ARE,FRED,0.26,integrated_circuit_components,OAC,1976,3
3,1976-04-01,100.0,ARE,FRED,0.26,integrated_circuit_components,OAC,1976,4
4,1976-05-01,100.0,ARE,FRED,0.26,integrated_circuit_components,OAC,1976,5


## Atttaching a representative unit value (average latest real_price for a product across the countries that are our company actually sources from_)

In [48]:
product_mapping = {
    'Microprocessors': 'microprocessors',
    'Power Devices': 'power_devices',
    'Transistors': 'transistors',
    'IC Components': 'integrated_circuit_components'
}

In [49]:
supplier_product['product'] = supplier_product['product'].map(product_mapping)

In [51]:
supplier_product.head()

,country_code,supplier_id,lead_time_mean,lead_time_variance,disruption_probability,compliance_eligibility,logistics_reliability,product,probability_of_defect,bulk_discount,bulk_units,baseline_price,price_volatility,hts8_tariff
0,ARE,SUP_ARE_1,29.081672,2.427785,0.283742,0.728946,0.844884,integrated_circuit_components,0.038476,0.092886,6892,0.246855,0.172129,85389030.0
1,AUS,SUP_AUS_2,53.948194,2.954602,0.200381,0.789128,0.779876,microprocessors,0.075537,0.026052,1139,0.991371,0.680152,NaN
2,AUS,SUP_AUS_3,53.655471,8.445529,0.226886,0.820285,0.768079,integrated_circuit_components,0.034397,0.104360,7775,0.213238,0.144877,85389030.0
3,BEL,SUP_BEL_4,30.632043,5.234186,0.274002,0.725275,0.795612,transistors,0.024267,0.085182,3714,0.044963,0.111905,NaN
4,BRA,SUP_BRA_6,40.588272,7.953977,0.482714,0.440989,0.678642,transistors,0.022972,0.087375,3455,0.040343,0.201362,NaN


In [58]:
latest_country_prices = (
    products.sort_values('date').groupby(
        ['country_code', 'product']
    ).real_price.last().reset_index())

In [60]:
# restricting to countries that actually have suppliers for the product

In [61]:
supplier_countries = supplier_product.groupby('product').country_code.unique()

In [62]:
def is_relevant(row):
  return row['country_code'] in supplier_countries.get(row['product'], [])

In [63]:
relevant_prices = latest_country_prices[latest_country_prices.apply(is_relevant, axis=1)]

In [65]:
# computing average over only those relevant countries

unit_values = (
    relevant_prices.groupby(
        'product').real_price.mean().to_dict())

In [66]:
demand_df.unit_value = demand_df['product'].map(unit_values)

In [68]:
demand_df.head()

,date,product,monthly_demand_units,unit_value
0,2024-01-01,microprocessors,16272,0.741299
1,2024-02-01,microprocessors,17746,0.741299
2,2024-03-01,microprocessors,17772,0.741299
3,2024-04-01,microprocessors,18981,0.741299
4,2024-05-01,microprocessors,16794,0.741299


# Holding Cost per unit per month

In [71]:
holding_pct = {
    'microprocessors': 0.025,
    'power_devices': 0.02,
    'transistors': 0.015,
    'integrated_circuit_components': 0.015
}

In [72]:
demand_df['unit_holding_cost_per_month'] = (
    demand_df.apply(lambda r: r['unit_value'] * holding_pct[r['product']], axis=1))

# Safety Stock, On-hand, On-Order units, and Backorder Units (Just in Case)

In [90]:
# converting lead time from days to months
lead_times_months = (
    supplier_product.groupby('product')['lead_time_mean'].median() / 30.0
).to_dict()

In [80]:
def safety_stock(row):
  lt_months = lead_times_months[row['product']]
  return int(row['monthly_demand_units'] * lt_months * 0.5) # 0.5x demand over lead time

In [81]:
demand_df['safety_stock_units'] = demand_df.apply(safety_stock, axis=1)

In [93]:
# On-hand = demand + safety stock +/- noise
def on_hand(row):
  base = row['monthly_demand_units'] + row['safety_stock_units']
  noise = np.random.uniform(0.8, 1.2)
  return int(base * noise)

In [94]:
demand_df['on_hand_units'] = demand_df.apply(on_hand, axis=1)

In [95]:
# On-order = 0.3-0.8x demand
demand_df['on_order_units'] = (
    demand_df['monthly_demand_units'] * np.random.uniform(
        0.3, 0.8, size=len(demand_df)).astype(int))

In [96]:
# Backorders (small and occassional)
demand_df['backorder_units'] = (
    (np.random.rand(len(demand_df)
    ) < 0.1) * np.random.randint(0, demand_df['monthly_demand_units'] * 0.2 + 1 )
).astype(int)

# Reorder Point (for EOQ/continuous review implementation if decide to include to factor this type of modeling technique into demo/presentation)

In [97]:
def reorder_point(row):
  lt_months = lead_times_months[row['product']]
  return int(row['monthly_demand_units'] * lt_months + row['safety_stock_units'])


demand_df['reorder_point_units'] = demand_df.apply(reorder_point, axis=1)

In [98]:
demand_df.columns

Index(['date', 'product', 'monthly_demand_units', 'unit_value',
       'unit_holding_cost_per_month', 'safety_stock_units', 'on_hand_units',
       'on_order_units', 'backorder_units', 'reorder_point_units'],
      dtype='object')

# Lead Time Months

The median lead time (in months) for that product across all suppliers who produce it

In [99]:
lead_times_months

{'integrated_circuit_components': 1.1287014696437412,
 'microprocessors': 1.1574783778869269,
 'power_devices': 1.133140445598098,
 'transistors': 1.092075627043478}

In [100]:
demand_df['lead_time_months'] = demand_df['product'].map(lead_times_months)

# Stockout Probability

Using a z-score approximation based on:
- on-hand
- expected demand during lead time
- safety stock

Produces:
- near-zero probability when on-hand is healthy
- rising probability as on-hand approaches reorder point
- high probability when on-hand < expected lead-time demand

In [101]:
from scipy.stats import norm

In [102]:
def stockout_prob(row):
  lt_months = row['lead_time_months']
  mu = row['monthly_demand_units'] * lt_months
  sigma = mu * 0.1 # demand noise = 10%

  # z-score for probabiity of NOT stocking out
  z = (row['on_hand_units'] - mu) / sigma if sigma > 0 else 10

  return float(1-norm.cdf(z))

demand_df['stockout_probability'] = demand_df.apply(stockout_prob, axis=1)

# Fixed ordering cost (K)

The cost of issuing a purchase order (any time an order is placed. Not impacted by order quantity)

Using a product-specific fixed ordering cost.

These are apparently industry benchmarks due to overhead, cost of supplier communication/negotiation, compliance chest, logistics coordination, etc.

In [103]:
ordering_cost = {
    'microprocessors': 1500,
    'power_devices': 1000,
    'transistors': 800,
    'integrated_circuit_components': 900
}

demand_df['fixed_ordering_cost'] = demand_df['product'].map(ordering_cost)

In [104]:
demand_df.head()

,date,product,monthly_demand_units,unit_value,unit_holding_cost_per_month,safety_stock_units,on_hand_units,on_order_units,backorder_units,reorder_point_units,lead_time_months,stockout_probability,fixed_ordering_cost
0,2024-01-01,microprocessors,16272,0.741299,0.018532,8446,26725,0,2366,27280,1.157478,1.398493e-05,1500
1,2024-02-01,microprocessors,17746,0.741299,0.018532,9212,31308,0,0,29752,1.157478,7.942263e-08,1500
2,2024-03-01,microprocessors,17772,0.741299,0.018532,9225,28222,0,0,29795,1.157478,9.980488e-05,1500
3,2024-04-01,microprocessors,18981,0.741299,0.018532,9853,23867,0,0,31823,1.157478,1.939582e-01,1500
4,2024-05-01,microprocessors,16794,0.741299,0.018532,8717,27448,0,0,28155,1.157478,1.891965e-05,1500


In [105]:
demand_df.to_csv('inventory_demand.csv', index=0)